[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/evals_quickstart.ipynb)


# Phoenix Evals 快速入门

本快速入门展示 Phoenix 如何帮助你评估 LLM 应用中的数据，例如输入、输出和检索文档。

你将：

- 从 Phoenix 会话中导出包含已插桩 LLM 应用追踪的 DataFrame。
- 评估追踪数据：
  - 相关性：检索到的文档是否为回答提供依据？
  - 问答正确性：应用回答是否以检索上下文为依据？
  - 忠实性：应用回答是否忠于检索上下文？
- 将评估导入 Phoenix，在对应的 span 和 trace 上查看结果标注。

开始吧！

首先，使用 `pip install arize-phoenix` 安装 Phoenix。

为了快速上手，我们将下载一些已从 LlamaIndex 应用收集的追踪数据。实际使用中，应通过兼容 OpenInference 的追踪器为 LLM 应用插桩来收集数据。# 待办：添加链接


In [ ]:
from urllib.request import urlopen

from phoenix.trace.trace_dataset import TraceDataset
from phoenix.trace.utils import json_lines_to_df

traces_url = "https://storage.googleapis.com/arize-phoenix-assets/datasets/unstructured/llm/context-retrieval/trace.jsonl"
with urlopen(traces_url) as response:
    lines = [line.decode("utf-8") for line in response.readlines()]
trace_ds = TraceDataset(json_lines_to_df(lines))

启动 Phoenix。可以在 Notebook 内使用，也可以打开 URL，在独立浏览器窗口中使用。


In [ ]:
import phoenix as px

session = px.launch_app(trace=trace_ds)
session.view()

现在应该看到如下视图。

![添加评估标注前的 Phoenix UI](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/evals/traces_without_evaluation_annotations.png)


将会话中的检索文档和查询数据导出到 pandas DataFrame。

注意：如需其他数据子集，可使用自定义查询导出。# 待办：添加链接


In [ ]:
from phoenix.client import Client
from phoenix.client.helpers.spans import get_input_output_context, get_retrieved_documents

retrieved_documents_df = get_retrieved_documents(Client())
queries_df = get_input_output_context(Client())

Phoenix 通过提示 LLM 对应用数据进行分类来评估，例如判断检索文档与对应查询相关还是不相关、回答是否以检索文档为依据等。你还可以让 LLM 生成解释，帮助理解评估结果！

本快速入门使用 OpenAI，需要 OpenAI API 密钥；我们也支持多种其他 API 和模型。# 待办：添加链接

使用 `pip install openai` 安装 OpenAI SDK 并实例化模型。


In [ ]:
from phoenix.evals.llm import LLM

api_key = None  # 在此处或通过 OPENAI_API_KEY 环境变量设置 API 密钥
eval_model = LLM(provider="openai", model="gpt-4o-mini", api_key=api_key)

接下来定义评估器。评估器构建在语言模型之上，通过提示 LLM 评判回答质量、检索文档相关性等，即使没有人工标注数据，也能提供质量信号。选择评估器类型，用希望执行评估的语言模型实例化，便可使用经过实战验证的评估模板。

![评估器由 LLM 和评估提示词模板组成，根据查询、参考资料、输出等输入数据生成标签、分数和解释](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/evals/evaluators_diagram.png)


In [ ]:
from phoenix.evals import bind_evaluator
from phoenix.evals.metrics import (
    CorrectnessEvaluator,
    FaithfulnessEvaluator,
    RetrievalRelevanceEvaluator,
)

faithfulness_evaluator = FaithfulnessEvaluator(eval_model)
qa_correctness_evaluator = CorrectnessEvaluator(eval_model)
relevance_evaluator = bind_evaluator(
    evaluator=RetrievalRelevanceEvaluator(eval_model),
    input_mapping={"context": "document"},
)

运行评估。


In [ ]:
import nest_asyncio
from phoenix.evals import async_evaluate_dataframe

nest_asyncio.apply()  # Notebook 环境中的并发所需

faithfulness_and_qa_df = await async_evaluate_dataframe(
    dataframe=queries_df,
    evaluators=[faithfulness_evaluator, qa_correctness_evaluator],
)

relevance_eval_df = await async_evaluate_dataframe(
    dataframe=retrieved_documents_df,
    evaluators=[relevance_evaluator],
)

将评估记录到正在运行的 Phoenix 会话。


In [ ]:
# 记录 span 评估
from phoenix.client import AsyncClient
from phoenix.evals.utils import to_annotation_dataframe

px_client = AsyncClient()

# 将评估结果转换为标注格式
faithfulness_and_qa_annotations = to_annotation_dataframe(faithfulness_and_qa_df)

await px_client.spans.log_span_annotations_dataframe(
    dataframe=faithfulness_and_qa_annotations,
    annotator_kind="LLM",
)

# 记录文档评估
relevance_annotations = to_annotation_dataframe(relevance_eval_df)

await px_client.spans.log_span_annotations_dataframe(
    dataframe=relevance_annotations,
    annotator_kind="LLM",
)

现在，评估应作为标注显示在 Phoenix 的 span 上！


In [ ]:
print(f"🔥🐦 如果已关闭 Phoenix，可重新打开： {session.url}")

你可以查看聚合评估统计，找出有问题的 span，阅读对应解释以理解 LLM 的评判理由，并定位 LLM 应用回答不佳的原因，例如检索无关或 LLM 参数设置不当。

![包含评估标注的 Phoenix UI](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/evals/traces_with_evaluation_annotations.png)
